In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import glob
import numpy as np
import matplotlib.pyplot as plt
import optuna
import copy
import random

class CNN2D(nn.Module):
    def _get_conv_output(self, shape):
        with torch.no_grad():
            x = torch.zeros(1, *shape)
            x = self.pool1(self.conv1(x))
            x = self.pool2(self.conv2(x))
            x = self.pool3(self.conv3(x))
            return x.numel()

    def __init__(self, input_channels, dropout_rate=0.6, fc_hidden=128):
        super(CNN2D, self).__init__()

        self.conv1 = nn.Sequential(
            nn.Conv2d(input_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU()
        )
        self.pool1 = nn.MaxPool2d(2, 2)

        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        self.pool2 = nn.MaxPool2d(2, 2)

        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU()
        )
        self.pool3 = nn.MaxPool2d(2, 2)

        self.flatten = nn.Flatten()
        conv_output_size = self._get_conv_output((input_channels, 200, 50))

        self.fc1 = nn.Linear(conv_output_size, fc_hidden)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(fc_hidden, 1)

    def forward(self, x):
        x = self.conv1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.pool2(x)

        x = self.conv3(x)
        x = self.pool3(x)

        x = self.flatten(x)
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

/home/alexhernandez/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Training method for PyTorch
def train_model(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    total = 0
    correct = 0

    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs).squeeze(1)
        loss = criterion(outputs, labels.squeeze(1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        preds = torch.sigmoid(outputs) >= 0.5
        correct += (preds == labels.squeeze(1).bool()).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    accuracy = correct / total
    return epoch_loss, accuracy

# Validation function
def validate_model(model, dataloader, criterion, device):
    model.eval()  # Set model to evaluation mode
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():  # Disable gradient calculation
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            # Forward pass
            outputs = model(inputs).squeeze(1)
            loss = criterion(outputs, labels.squeeze(1))
            running_loss += loss.item() * inputs.size(0)
            
            # Compute accuracy
            preds = torch.sigmoid(outputs) >= 0.5
            correct += (preds == labels.squeeze(1).bool()).sum().item()
            total += labels.size(0)
    
    validation_loss = running_loss / total
    accuracy = correct / total
    return validation_loss, accuracy

In [3]:
class GridDataset(Dataset):
    def __init__(self, data_dict):
        self.data = list(data_dict.values())

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        grid = sample['grid_tensor']
        label = torch.tensor(sample['label'], dtype=torch.float32)
        return grid, label.unsqueeze(0)

In [4]:
# Create a dictionary with file names as keys and label + tensor grid as values
positive_grids = glob.glob('/home/alexhernandez/CholBindNet/CLR_Ligand_Data/cholesterol-rdkit-fpocket-5A_exp1/PositiveWithoutSpies/*.npy')
validation_grids = glob.glob('/home/alexhernandez/CholBindNet/CLR_Ligand_Data/cholesterol-rdkit-fpocket-5A_exp1/Validation_Set/*.npy')
file_data = {} # format is filename as key, label and grid tensor are values

for file in positive_grids:
    # Load the numpy array and convert it to a PyTorch tensor
    grid = np.load(file)
    grid_tensor = torch.tensor(grid, dtype=torch.float32).unsqueeze(0)  # Adds channel dimension
    file_data[file] = {'label': 1, 'grid_tensor': grid_tensor}
positive_grids = file_data
print(len(positive_grids), "is length of positive")

file_data = {} # format is filename as key, label and grid tensor are values

positive_validation_count = 0
unlabeled_validation_count = 0

import random

positive_files = []
unlabeled_files = []

# Step 1: Separate files
for file in validation_grids:
    if any(f"-p{i}" in file for i in range(1, 999)):
        unlabeled_files.append(file)
    else:
        positive_files.append(file)

print("Before balancing:")
print("Positives:", len(positive_files))
print("Unlabeled:", len(unlabeled_files))

# Step 2: Balance counts
min_count = min(len(positive_files), len(unlabeled_files))

positive_files = random.sample(positive_files, min_count)
unlabeled_files = random.sample(unlabeled_files, min_count)

balanced_files = positive_files + unlabeled_files
random.shuffle(balanced_files)

# Step 3: Process balanced set
file_data = {}
positive_validation_count = 0
unlabeled_validation_count = 0

for file in balanced_files:
    grid = np.load(file)
    grid_tensor = torch.tensor(grid, dtype=torch.float32).unsqueeze(0)

    if file in unlabeled_files:
        label = 0
        unlabeled_validation_count += 1
    else:
        label = 1
        positive_validation_count += 1

    file_data[file] = {'label': label, 'grid_tensor': grid_tensor}

print("After balancing:")
print("Positives:", positive_validation_count)
print("Unlabeled:", unlabeled_validation_count)

validation_grids = file_data
print(len(validation_grids), "is length of validation grids")


file_data = {} # format is filename as key, label and grid tensor are values

k = 50
subset_grids = []
for i in range(1, k + 1):
    file_data = {}
    subset_grid = glob.glob(f'/home/alexhernandez/CholBindNet/CLR_Ligand_Data/cholesterol-rdkit-fpocket-5A_exp1/k_subsets/subset_{i}/*.npy')  # Adjust path as needed
    for file in subset_grid:
        # Load the numpy array and convert it to a PyTorch tensor
        grid = np.load(file)
        grid_tensor = torch.tensor(grid, dtype=torch.float32).unsqueeze(0)  # Adds channel dimension
        file_data[file] = {'label': 0, 'grid_tensor': grid_tensor} # 0 means unlabeled
    subset_grid = file_data
    subset_grids.append(subset_grid)
    print(len(subset_grid), "is length of subset grid")

bins = []
for subset_grid in subset_grids:
    bin = {**positive_grids, **subset_grid} # merged
    bins.append(bin)


385 is length of positive
Before balancing:
Positives: 77
Unlabeled: 5659
After balancing:
Positives: 77
Unlabeled: 77
154 is length of validation grids
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is length of subset grid
385 is

In [5]:
def plot_graphs(train_losses, validation_losses, validation_accuracies, learning_rates):
    # Plot Training Loss vs Validation Loss
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(train_losses) + 1), train_losses, label='Training Loss')
    plt.plot(range(1, len(validation_losses) + 1), validation_losses, label='Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title('Training Loss vs Validation Loss')
    plt.legend()
    plt.grid()
    plt.show()

    # Plot Validation Accuracy
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(validation_accuracies) + 1), validation_accuracies, label='Validation Accuracy', color='green')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.title('Validation Accuracy over Epochs')
    plt.legend()
    plt.grid()
    plt.show()

    # Plot Learning Rate
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(learning_rates) + 1), learning_rates, label='Learning Rates', color='green')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.title('Validation Accuracy over Epochs')
    plt.legend()
    plt.grid()
    plt.show()

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def objective(trial):
    # Reproducibility
    seed = 42
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Hyperparameters to tune
    lr = trial.suggest_float("lr", 5e-7, 5e-5, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-4, 1e-2, log=True)
    dropout_rate = trial.suggest_float("dropout_rate", 0.45, 0.75)
    batch_size = trial.suggest_categorical("batch_size", [16, 32])
    fc_hidden = trial.suggest_categorical("fc_hidden", [32, 64, 128])

    # Tuning settings
    tune_epochs = 1000
    num_bins_to_use = 1

    # Use only the first bin
    selected_bin_indices = list(range(min(num_bins_to_use, len(bins))))

    criterion = nn.BCEWithLogitsLoss()
    bin_val_losses = []

    validation_dataset = GridDataset(validation_grids)
    validation_loader = DataLoader(validation_dataset, batch_size=batch_size, shuffle=False)

    for bin_idx in selected_bin_indices:
        model = CNN2D(
            input_channels=1,
            dropout_rate=dropout_rate,
            fc_hidden=fc_hidden
        ).to(device)

        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

        train_dataset = GridDataset(bins[bin_idx])
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

        best_val_loss = float("inf")

        for epoch in range(tune_epochs):
            train_loss, train_acc = train_model(model, train_loader, criterion, optimizer, device)
            val_loss, val_acc = validate_model(model, validation_loader, criterion, device)

            # Track best validation loss
            if val_loss < best_val_loss:
                best_val_loss = val_loss

            # Report intermediate value for pruning
            trial.report(val_loss, step=epoch)

            if trial.should_prune():
                raise optuna.TrialPruned()

            if epoch % 50 == 0 or epoch == tune_epochs - 1:
                print(
                    f"Trial {trial.number}, Bin {bin_idx+1}, Epoch {epoch+1}/{tune_epochs}, "
                    f"Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, "
                    f"Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}"
                )

        bin_val_losses.append(val_loss)

    # Minimize average best validation loss
    return float(np.mean(bin_val_losses))


sampler = optuna.samplers.TPESampler(seed=42)
pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=20)

study = optuna.create_study(
    direction="minimize",
    sampler=sampler,
    pruner=pruner
)

study.optimize(objective, n_trials=30)

print("Best trial:")
print("  Value:", study.best_trial.value)
print("  Params:")
for k, v in study.best_trial.params.items():
    print(f"    {k}: {v}")

best_params = study.best_trial.params
print(best_params)

[I 2026-04-17 11:35:23,716] A new study created in memory with name: no-name-e607d3de-f198-4ba2-93aa-282431702769


Trial 0, Bin 1, Epoch 1/1000, Train Loss: 0.7006, Val Loss: 0.6651, Train Acc: 0.5273, Val Acc: 0.5390
Trial 0, Bin 1, Epoch 51/1000, Train Loss: 0.4585, Val Loss: 0.4523, Train Acc: 0.7779, Val Acc: 0.7662
Trial 0, Bin 1, Epoch 101/1000, Train Loss: 0.3800, Val Loss: 0.4349, Train Acc: 0.8403, Val Acc: 0.7792
Trial 0, Bin 1, Epoch 151/1000, Train Loss: 0.3199, Val Loss: 0.4215, Train Acc: 0.8662, Val Acc: 0.8052
Trial 0, Bin 1, Epoch 201/1000, Train Loss: 0.2603, Val Loss: 0.4081, Train Acc: 0.9078, Val Acc: 0.8182
Trial 0, Bin 1, Epoch 251/1000, Train Loss: 0.2032, Val Loss: 0.3994, Train Acc: 0.9286, Val Acc: 0.8247
Trial 0, Bin 1, Epoch 301/1000, Train Loss: 0.1683, Val Loss: 0.4087, Train Acc: 0.9558, Val Acc: 0.8377
Trial 0, Bin 1, Epoch 351/1000, Train Loss: 0.1223, Val Loss: 0.4056, Train Acc: 0.9701, Val Acc: 0.8442
Trial 0, Bin 1, Epoch 401/1000, Train Loss: 0.0973, Val Loss: 0.4186, Train Acc: 0.9818, Val Acc: 0.8377
Trial 0, Bin 1, Epoch 451/1000, Train Loss: 0.0727, Val Lo

[I 2026-04-17 11:41:45,609] Trial 0 finished with value: 0.7480523597884488 and parameters: {'lr': 2.8057582076672515e-06, 'weight_decay': 0.007969454818643935, 'dropout_rate': 0.6695981825434215, 'batch_size': 16, 'fc_hidden': 128}. Best is trial 0 with value: 0.7480523597884488.


Trial 0, Bin 1, Epoch 1000/1000, Train Loss: 0.0032, Val Loss: 0.7481, Train Acc: 1.0000, Val Acc: 0.8312
Trial 1, Bin 1, Epoch 1/1000, Train Loss: 0.6795, Val Loss: 0.6476, Train Acc: 0.5714, Val Acc: 0.6039
Trial 1, Bin 1, Epoch 51/1000, Train Loss: 0.3607, Val Loss: 0.4154, Train Acc: 0.8351, Val Acc: 0.7857
Trial 1, Bin 1, Epoch 101/1000, Train Loss: 0.2083, Val Loss: 0.3992, Train Acc: 0.9390, Val Acc: 0.8442
Trial 1, Bin 1, Epoch 151/1000, Train Loss: 0.1164, Val Loss: 0.3990, Train Acc: 0.9740, Val Acc: 0.8506
Trial 1, Bin 1, Epoch 201/1000, Train Loss: 0.0599, Val Loss: 0.4445, Train Acc: 0.9909, Val Acc: 0.8377
Trial 1, Bin 1, Epoch 251/1000, Train Loss: 0.0289, Val Loss: 0.5198, Train Acc: 0.9961, Val Acc: 0.8571
Trial 1, Bin 1, Epoch 301/1000, Train Loss: 0.0140, Val Loss: 0.5819, Train Acc: 0.9987, Val Acc: 0.8636
Trial 1, Bin 1, Epoch 351/1000, Train Loss: 0.0078, Val Loss: 0.6551, Train Acc: 1.0000, Val Acc: 0.8377
Trial 1, Bin 1, Epoch 401/1000, Train Loss: 0.0033, Val L

[I 2026-04-17 11:48:06,301] Trial 1 finished with value: 0.971083184389712 and parameters: {'lr': 7.965261308120506e-06, 'weight_decay': 0.0026070247583707684, 'dropout_rate': 0.45617534828874073, 'batch_size': 16, 'fc_hidden': 32}. Best is trial 0 with value: 0.7480523597884488.


...

Trial 28, Bin 1, Epoch 1/1000, Train Loss: 0.6821, Val Loss: 0.6479, Train Acc: 0.5468, Val Acc: 0.5584


[I 2026-04-17 13:05:13,259] Trial 28 pruned. 


Trial 29, Bin 1, Epoch 1/1000, Train Loss: 0.6900, Val Loss: 0.6567, Train Acc: 0.5364, Val Acc: 0.6299


[I 2026-04-17 13:05:20,707] Trial 29 pruned. 


Best trial:
  Value: 0.3797588940564688
  Params:
    lr: 2.7800474932883292e-05
    weight_decay: 0.009734059625918745
    dropout_rate: 0.6221988693502027
    batch_size: 16
    fc_hidden: 128
{'lr': 2.7800474932883292e-05, 'weight_decay': 0.009734059625918745, 'dropout_rate': 0.6221988693502027, 'batch_size': 16, 'fc_hidden': 128}


In [7]:
# import os

# # Set device
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# # Initialize the model
# criterion = nn.BCEWithLogitsLoss()

# # Define paths for saving models
# save_dir = "RDKIT-GNN-5A_Exp1"
# os.makedirs(save_dir, exist_ok=True)

# # Training loop
# epochs = 2000
# batch_size = 8

# # keep 10 positives and 10 negatives for validation data
# validation_dataset = GridDataset(validation_grids)
# validation_dataloader = DataLoader(validation_dataset, batch_size=batch_size, shuffle=False)

# for i in range(0, len(bins)):
#     model = CNN2D(input_channels=1).to(device)
#     optimizer = optim.Adam(model.parameters(), lr=0.000001, weight_decay=1e-3) 
#     scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5000, eta_min=1e-10)

#     print(f"Training on bin {i+1}/{len(bins)}")
#     dataset = GridDataset(bins[i])
#     dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

#     train_losses = []
#     learning_rates = []
#     validation_losses = []
#     validation_accuracies = []

#     best_val_loss = float("inf")
#     best_model_state = None

#     for epoch in range(epochs):
#         epoch_loss, accuracy = train_model(model, dataloader, criterion, optimizer, device)
#         validation_loss, validation_accuracy = validate_model(model, validation_dataloader, criterion, device)
#         current_lr = optimizer.param_groups[0]['lr'] 
#         train_losses.append(epoch_loss)
#         learning_rates.append(current_lr)
#         validation_losses.append(validation_loss)
#         validation_accuracies.append(validation_accuracy)   
#         if epoch % 10 == 0:
#             print(
#                 f"Bin {i+1}, Epoch {epoch+1}/{epochs}, "
#                 f"Train Loss: {epoch_loss:.4f}, Validation Loss: {validation_loss:.4f},  "
#                 f"Accuracy: {validation_accuracy:.4f}, "
#                 f"LR: {current_lr:.6f}"
#             )

#         if validation_loss < best_val_loss:
#             best_val_loss = validation_loss
#             best_model_state = model.state_dict()
#         scheduler.step()

#     plot_graphs(train_losses, validation_losses, validation_accuracies, learning_rates)
    
#     #Save the trained model
#     model_path = os.path.join(save_dir, f"model_bin_{i+1}.pth")
#     torch.save(model.state_dict(), model_path)
#     print(f"Model for bin {i+1} saved to {model_path}")


# print("Training complete.")

